In [11]:
import re
from pathlib import Path

# Regex pattern
pattern = re.compile(r"(?<!!)\[\[([^|\]]+?)#([^\]|]+?)\]\]")


def get_last_header(header_chain):
    """
    Extract the last header from a chain like:
    'Header1#Header2#Header3' -> 'Header3'
    """
    return header_chain.split("#")[-1].strip()

def process_file(filepath):
    text = filepath.read_text(encoding="utf-8")
    matches = list(pattern.finditer(text))

    if not matches:
        return

    print(f"\n--- {filepath} ---")

    new_text = text
    offset = 0
    apply_all = True

    for match in matches:
        page = match.group(1).strip()
        header_chain = match.group(2).strip()

        # Skip if already has alias (extra safety)
        if "|" in header_chain:
            continue

        alias = get_last_header(header_chain)

        original = match.group(0)
        replacement = f"[[{page}#{header_chain}|{alias}]]"

        print(f"\nFound:      {original}")
        print(f"Replace ->  {replacement}")

        if not apply_all:
            choice = input("Replace? (y = yes, n = no, a = all, q = quit): ").lower()

            if choice == "q":
                print("Exiting.")
                exit()
            elif choice == "n":
                continue
            elif choice == "a":
                apply_all = True
            elif choice != "y":
                continue

        # Apply replacement (account for shifting indices)
        start, end = match.start() + offset, match.end() + offset
        new_text = new_text[:start] + replacement + new_text[end:]
        offset += len(replacement) - (end - start)

    if new_text != text:
        filepath.write_text(new_text, encoding="utf-8")
        print("✔ File updated")


def scan_directory(root):
    for path in Path(root).rglob("*.md"):
        print(f"Scanning: {path}")
        process_file(path)


if __name__ == "__main__":
    vault_path = "/Users/ayelland/ObsidianVaults/AstroWiki_2.0/" #input("Enter path to your Obsidian vault: ").strip()
    scan_directory(vault_path)
    print("\nDone.")

Scanning: /Users/ayelland/ObsidianVaults/AstroWiki_2.0/Mnemonics.md

--- /Users/ayelland/ObsidianVaults/AstroWiki_2.0/Mnemonics.md ---

Found:      [[_Stellar Nucleosynthesis#CNO Cycle]]
Replace ->  [[_Stellar Nucleosynthesis#CNO Cycle|CNO Cycle]]

Found:      [[Velocity dispersion#Tully-Fisher Relation]]
Replace ->  [[Velocity dispersion#Tully-Fisher Relation|Tully-Fisher Relation]]

Found:      [[Velocity Dispersion#Faber-Jackson Relation]]
Replace ->  [[Velocity Dispersion#Faber-Jackson Relation|Faber-Jackson Relation]]

Found:      [[Kepler's Laws of Planetary Motion#Kepler's 3rd Law]]
Replace ->  [[Kepler's Laws of Planetary Motion#Kepler's 3rd Law|Kepler's 3rd Law]]

Found:      [[Galaxy Profiles#Einasto Profile]]
Replace ->  [[Galaxy Profiles#Einasto Profile|Einasto Profile]]
✔ File updated
Scanning: /Users/ayelland/ObsidianVaults/AstroWiki_2.0/AstroWiki Homepage.md
Scanning: /Users/ayelland/ObsidianVaults/AstroWiki_2.0/168 Flashcards.md
Scanning: /Users/ayelland/ObsidianVaults/